In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from quick_pp.database.objects import Project
from quick_pp.database.db_connector import DBConnector

db_conn = DBConnector()

# Load well from saved file
project_name = "30-7a"
well_name = "30-7a-2"
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    all_data = project.get_all_data()
    well_data = project.get_well_data(well_name)

# Filter to sandshale only
all_data = all_data[all_data["model"] == "sandshale"].copy()

***
## Leverett J Method using FZI Rock Types

In [ ]:
import numpy as np

32 * np.cos(np.radians(0))

In [ ]:
import pandas as pd

from quick_pp.rock_type import calc_fzi, rock_typing, plot_fzi
from quick_pp.core_analysis import (
    fit_j_curve,
    j_xplot,
    leverett_j,
    sw_shf_leverett_j,
    poroperm_xplot,
    pc_xplot,
)

from quick_pp.core_analysis import (
    restructure_scal_data,
    string_to_int_hash,
    auto_cluster_scal_data,
)

raw_core_data = pd.read_excel(rf"data\{project_name}_SCAL.xlsx")
core_data = restructure_scal_data(raw_core_data)

# Filter to sandshale only
core_data = core_data[core_data["Model"] == "sandshale"].copy()

core_data["Sample"] = core_data["Sample ID"]
core_data["SampleID"] = core_data["Sample ID"].apply(string_to_int_hash)
core_data["CPORE"] = core_data["PHI_frac"]
core_data["CPERM"] = core_data["K_mD"]
core_data["PC"] = core_data["Pc"]
core_data["PC_RES"] = core_data["PC"] * (26 / 72)
core_data["SW"] = core_data["Sw"]
core_data["SWN"] = core_data.groupby(["Well", "Sample"])["SW"].transform(
    lambda x: (x - x.min()) / (1 - x.min())
)


# Calculate J
ift = 26
theta = 0

core_data["J"] = leverett_j(
    core_data["PC_RES"], ift, theta, core_data["CPERM"], core_data["CPORE"]
)

# Plot J
j_xplot(core_data["SWN"], core_data["J"], ylim=(0, 15))

In [ ]:
import json
from quick_pp.rock_type import plot_fzi

# Load FZI cutoffs
with open(rf"data\04_project\{project_name}\outputs\fzi_cutoffs.json", "rb") as file:
    fzi_cutoffs = json.load(file)

# FZI
fzi = calc_fzi(core_data["CPORE"], core_data["CPERM"])
rock_flag = rock_typing(fzi, fzi_cutoffs, higher_is_better=True)
core_data["ROCK_FLAG"] = rock_flag

core_data = auto_cluster_scal_data(core_data)

plot_fzi(
    core_data["CPORE"],
    core_data["CPERM"],
    rock_type=core_data["ROCK_FLAG"],
    cut_offs=fzi_cutoffs,
)
print(pd.Series(rock_flag).value_counts().sort_index())

In [ ]:
from quick_pp.core_analysis import poroperm_xplot

poroperm_xplot(core_data.CPORE, core_data.CPERM, core_group=core_data.CORE_CLUSTER)

In [ ]:
from quick_pp.core_analysis import plot_pc_by_prt

plot_pc_by_prt(core_data, 80)

In [ ]:
# Plot PTSD distribution
from quick_pp.core_analysis import plot_ptsd_by_prt

copy_df = plot_ptsd_by_prt(core_data, ift, theta)

In [ ]:
from ipywidgets import interact, widgets
import plotly.graph_objects as go

from quick_pp.core_analysis import pc_xplot, poroperm_xplot, pc_xplot_plotly

rock_flag_widget = widgets.SelectMultiple(
    options=["All"] + sorted(list(core_data["ROCK_FLAG"].unique())),
    value=["All"],
    description="Rock Flag:",
)


@interact(rock_flag=rock_flag_widget)
def param(rock_flag):
    # Plot all data on poroperm plot
    poroperm_xplot(core_data["CPORE"], core_data["CPERM"])
    data = (
        core_data[core_data.ROCK_FLAG.isin(rock_flag)]
        if any([l for l in rock_flag if l != "All"])
        else core_data
    )

    # Plot filtered data
    poroperm_data = data.drop_duplicates(subset=["CPORE", "CPERM"], keep="last")
    poroperm_xplot(
        poroperm_data["CPORE"],
        poroperm_data["CPERM"],
        core_group=poroperm_data["SampleID"],
    )
    plt.show()

    fig = go.Figure()
    for label, temp_df in data.groupby(["Well", "Sample"]):
        temp_df = temp_df.sort_values("PC_RES").copy()
        fig = pc_xplot_plotly(
            temp_df["SWN"], temp_df["PC_RES"], label=str(label), ylim=(0, 20), fig=fig
        )
    fig.show()


plt.close("all")

#### QC the Pc data

The capillary pressure measurements for each Sample are plotted on a log-log plot.
The data points should fall on a relatively straight line indicating good data quality.

Based 
select the dataset for each rock type
curve fitting

In [ ]:
import json
from quick_pp.core_analysis import auto_j_params

mapped_fzi_params = auto_j_params(core_data, cluster_by="CORE_CLUSTER")

with open(
    rf"data\04_project\{project_name}\outputs\rt_j_params.json", "w", encoding="utf-8"
) as json_file:
    json.dump(mapped_fzi_params, json_file, ensure_ascii=False, indent=4)

mapped_fzi_params

In [ ]:
from ipywidgets import interact, widgets

rt_widget = widgets.Dropdown(
    options=sorted(core_data["ROCK_FLAG"].unique()), description="Rock Type:"
)


@interact(rt=rt_widget)
def param(rt):
    params = next(item for item in mapped_fzi_params if item["ROCK_FLAG"] == rt)
    a, b = params["a"], params["b"]
    data = core_data[core_data["ROCK_FLAG"] == rt]

    j_xplot(
        data["SWN"],
        data["J"],
        a=a,
        b=b,
        core_group=data["Sample"],
        log_log=True,
        ylim=(0, 10),
        label=f"Rock Type {rt}: a:{a}, b:{b}",
    )

In [ ]:
from quick_pp.core_analysis import plot_j_by_prt

plot_j_by_prt(core_data, mapped_fzi_params, ymax=10)

In [ ]:
import numpy as np

from quick_pp.utils import inv_power_law_func

# Plot mapped_fzi_params on the same j_xplot
for rock in core_data.ROCK_FLAG.unique():
    # Extract values from the dictionary 'd'
    params = next(item for item in mapped_fzi_params if item["ROCK_FLAG"] == rock)
    a, b = params["a"], params["b"]
    csw = np.geomspace(0.01, 1.0, 50)
    plt.plot(
        csw,
        inv_power_law_func(csw, a, b),
        label=f"PRT {rock}, a: {a}, b: {b}",
        linestyle="dashed",
    )
plt.title("J Curve vs SW")
plt.xlabel("SW (v/v)")
plt.ylabel("J (unitless)")
plt.xlim(0, 1)
plt.ylim(0, 2)
plt.legend()

#### Estimate Free Water Level (FWL)

In [ ]:
import numpy as np
import pickle
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer


from quick_pp.rock_type import calc_fzi_perm

imp_mean = SimpleImputer(missing_values=np.nan, strategy="mean")

# Predict ROCK_FLAG
input_features = ["GR", "RHOB", "VCLAY", "PHIE", "PGF", "NPHI"]
with open(rf"data\04_project\{project_name}\outputs\fzi_rt_model.qppm", "rb") as file:
    fzi_rt_model = pickle.load(file)
well_data["ROCK_FLAG"] = fzi_rt_model.predict(well_data[input_features])

# Predict PERM
temp_df = well_data.copy()
temp_df["ROCK_PRED"] = well_data["ROCK_FLAG"]
input_features_fzi = input_features + ["ROCK_PRED"]
temp_df[input_features] = imp_mean.fit_transform(temp_df[input_features])
with open(rf"data\04_project\{project_name}\outputs\fzi_model.qppm", "rb") as file:
    fzi_model = pickle.load(file)
fzi_ml = 10 ** (fzi_model.predict(temp_df[input_features_fzi]))
well_data["PERM"] = calc_fzi_perm(fzi_ml, well_data["PHIT"])


In [ ]:
well_data.columns

In [ ]:
from ipywidgets import interactive, widgets
from quick_pp.saturation import *
from quick_pp.core_analysis import sw_shf_leverett_j

# Debug water saturation
water_salinity = 160e3
m = 1.9

ift = 32
theta = 30
ghc = 0.837
gw = 1.135
fwl = 2960

fwl = widgets.FloatSlider(value=fwl, min=fwl / 1.1, max=fwl * 1.1, step=1)


def plot(fwl):
    # Create a mapping from ROCK_FLAG to its 'a' and 'b' parameters for efficient lookup.
    # This is more robust than iterating through the list for each row.
    param_map = {item["ROCK_FLAG"]: item for item in mapped_fzi_params}

    # Map the ROCK_FLAG in well_data to the corresponding 'a' and 'b' values.
    a = well_data["ROCK_FLAG"].map(lambda x: param_map.get(x, {}).get("a"))
    b = well_data["ROCK_FLAG"].map(lambda x: param_map.get(x, {}).get("b"))
    shf = sw_shf_leverett_j(
        well_data["PERM"],
        well_data["PHIT"],
        well_data["TVD"],
        gw=gw,
        ghc=ghc,
        fwl=fwl,
        ift=ift,
        theta=theta,
        a=a,
        b=b,
    )

    temp_grad = estimate_temperature_gradient(well_data["DEPTH"], "imperial")
    rw = estimate_rw_temperature_salinity(temp_grad, water_salinity)

    swt = archie_saturation(well_data["RT"], rw, well_data["PHIT"], m=m)
    swt = swt.clip(0, 1)

    bvo = well_data["PHIT"] * (1 - swt)
    bvoh = well_data["PHIT"] * (1 - shf)

    # Create 3 subplots that share the x-axis
    fig, axes = plt.subplots(3, 1, figsize=(20, 3), sharex=True)
    fig.suptitle(f"Water Saturation for {well_name}", fontsize=16)

    # Plot 1: Water Saturation
    axes[0].plot(well_data["DEPTH"], swt, label="SWT")
    axes[0].plot(well_data["DEPTH"], shf, label="SHF")
    axes[0].plot(
        well_data["DEPTH"], np.ones(len(well_data)), color="black", linestyle="--"
    )
    axes[0].set_ylim(0, 1.2)
    axes[0].legend()
    # Plot 2: Bulk Volume Oil
    axes[1].plot(well_data["DEPTH"], bvo, label="BVO")
    axes[1].plot(well_data["DEPTH"], bvoh, label=r"$BVO_{SHF}$")
    axes[1].set_ylim(0, 0.5)
    axes[1].legend()

    # Plot 3: Rock Flag
    axes[2].plot(well_data["DEPTH"], well_data["ROCK_FLAG"], label="Rock Flag")
    axes[2].legend()
    axes[2].set_xlim(fwl - 150, fwl + 150)


interactive_plot = interactive(plot, fwl=fwl)
output = interactive_plot.children[-1]
output.layout.height = "350px"
interactive_plot

***
## Log Derived Water Saturation

In [ ]:
from ipywidgets import widgets, interact

from quick_pp.saturation import pickett_plot

focused_data = all_data.copy()

wells = widgets.SelectMultiple(
    options=["All"] + list(focused_data["WELL_NAME"].unique()),
    value=["All"],
    description="Wells:",
)
m = widgets.FloatSlider(value=2, min=1, max=5, step=0.1, readout_format=".1f")
min_rw = widgets.FloatSlider(
    value=0.01, min=0.001, max=1.0, step=0.001, readout_format=".3f"
)
min_depth = widgets.FloatSlider(
    value=focused_data.DEPTH.min(),
    min=focused_data.DEPTH.min(),
    max=focused_data.DEPTH.max() - 10,
    step=0.1,
    readout_format=".1f",
)
max_depth = widgets.FloatSlider(
    value=focused_data.DEPTH.max(),
    min=focused_data.DEPTH.min() + 10,
    max=focused_data.DEPTH.max(),
    step=0.1,
    readout_format=".1f",
)


@interact(wells=wells, m=m, min_rw=min_rw, min_depth=min_depth, max_depth=max_depth)
def param(wells, m, min_rw, min_depth, max_depth):
    if "All" in wells:
        data = focused_data[
            (focused_data.DEPTH >= min_depth) & (focused_data.DEPTH <= max_depth)
        ]
    else:
        data = focused_data[
            (focused_data.WELL_NAME.isin(wells))
            & (focused_data.DEPTH >= min_depth)
            & (focused_data.DEPTH <= max_depth)
        ]
    pickett_plot(
        data["RT"],
        data["PHIT"],
        m=m,
        min_rw=min_rw,
        title=f"Pickett Plot for {wells[0]}",
    )

In [ ]:
import numpy as np
from matplotlib import ticker as mticker

from quick_pp.saturation import *
from quick_pp.porosity import *

water_salinity = 100e3
m = 2

temp_grad = estimate_temperature_gradient(well_data["TVD"], "metric")
rw = estimate_rw_temperature_salinity(temp_grad, water_salinity)
b = estimate_b_waxman_smits(temp_grad, rw)
qv = estimate_qv(well_data.VCLAY, well_data.PHIT, cec_clay=0.1)
phit_shale = estimate_shale_porosity(well_data.NPHI, well_data.PHIT)
rt_shale = estimate_rt_shale(well_data.RT, well_data.VCLAY)
qvn = estimate_qvn(well_data.VCLAY, well_data.PHIT, phit_shale)

swt_ws = waxman_smits_saturation(well_data["RT"], rw, well_data.PHIE, B=b, Qv=qvn, m=m)
swt_nws = normalized_waxman_smits_saturation(
    well_data.RT, rw, well_data.PHIT, well_data.VCLAY, phit_shale, rt_shale=8, m=m
)

swt_archie = archie_saturation(well_data.RT, rw, well_data.PHIT, m=m)

fig, axes = plt.subplots(3, 1, figsize=(15, 5), sharex=True)
axes[0].plot(well_data["DEPTH"], swt_nws, label="SWT Normalized WS")
axes[0].plot(well_data["DEPTH"], swt_ws, label="SWT WS")
axes[0].plot(well_data["DEPTH"], swt_archie, label="SWT Archie")
axes[0].plot(well_data["DEPTH"], np.ones(len(well_data)), color="black", linestyle="--")
axes[0].set_ylim(0, 2)
axes[0].legend()

axes[1].plot(well_data["DEPTH"], rw, label="RW")
axes[1].set_yscale("log")
axes[1].yaxis.set_minor_formatter(mticker.ScalarFormatter())
axes[1].legend()

axes[2].plot(well_data["DEPTH"], temp_grad, label="Temperature")
axes[2].legend()

fig.tight_layout()

***
# Plot the results

In [ ]:
from quick_pp.plotter.plotter import plotly_log
from quick_pp.core_analysis import *

# Plot individual results
well_data["SWT"] = swt_ws
fig = plotly_log(well_data, well_name=well_name, depth_uom="m")
fig.show(config=dict(scrollZoom=True))

# Apply to all

In [ ]:
from tqdm import tqdm

from quick_pp.saturation import *
from quick_pp.porosity import *
from quick_pp.core_analysis import sw_shf_leverett_j


water_salinity = 100e3
m = 2

ift = 26
theta = 0
ghc = 0.837
gw = 1.135
fwl = 2690

for well_name, plot_data in tqdm(all_data.groupby("WELL_NAME")):
    tqdm.write(f"Processing {well_name}: {len(plot_data)} rows")

    # Log derived - Waxman-Smits
    temp_grad = estimate_temperature_gradient(plot_data["TVD"], "metric")
    rw = estimate_rw_temperature_salinity(temp_grad, water_salinity)
    b = estimate_b_waxman_smits(temp_grad, rw)
    phit_shale = estimate_shale_porosity(plot_data.NPHI, plot_data.PHIT)
    qvn = estimate_qvn(plot_data.VCLAY, plot_data.PHIT, phit_shale)
    swt = waxman_smits_saturation(plot_data["RT"], rw, plot_data.PHIE, B=b, Qv=qvn, m=m)

    # Saturation height function - Leverett-J
    for rock, rock_data in plot_data.groupby("ROCK_FLAG"):
        if rock < plot_data.ROCK_FLAG.nunique():
            params = next(
                item for item in mapped_fzi_params if item["ROCK_FLAG"] == rock
            )
            a, b = params["a"], params["b"]
            shf = sw_shf_leverett_j(
                rock_data["PERM"],
                rock_data["PHIT"],
                rock_data["TVD"],
                gw=gw,
                ghc=ghc,
                fwl=fwl,
                ift=ift,
                theta=theta,
                a=a,
                b=b,
            )
            plot_data.loc[rock_data.index, "SHF"] = shf.clip(0, 1)

    plot_data["SWT"] = swt.clip(0, 1)

    # Save result to database
    with db_conn.get_session() as db_session:
        project = Project(db_session, name=project_name)
        project.update_data(plot_data)
        project.save()